# Deep Neural Network Approximation of the Black-Scholes Pricing Surface
### M2R Group 51 — Week 3 | Justin

---

## 1. Introduction and Motivation

The Black-Scholes-Merton (BSM) model provides a closed-form solution for the price of a European call option under the assumptions of log-normally distributed asset returns and constant volatility. For a European call, the price is given by:

$$C(S, K, T, r, \sigma) = S \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

where
$$d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

and $\Phi(\cdot)$ is the standard normal CDF.

Although BSM has an analytical solution, it serves as the ideal benchmark for validating a neural network approximation approach. More complex models — such as Heston stochastic volatility or SABR — do **not** admit closed-form solutions and require expensive Monte Carlo or PDE-based numerical solvers. A trained DNN can price options in microseconds, making it an attractive surrogate model in calibration routines that require millions of evaluations.

**This notebook trains a feedforward Deep Neural Network (DNN) to learn the Black-Scholes pricing map $f: (S, K, T, r, \sigma) \mapsto C$, benchmarks its accuracy, and quantifies the computational speedup it delivers.**

---
## 2. Imports and Environment Setup

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Project imports
sys.path.append('../../')
from src.pricing.models import bsm_european

# Numerical / scientific
import numpy as np
import pandas as pd

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

---
## 3. Data Generation — Analytical Ground Truth

Because the BSM formula is exact, we generate training data with **zero variance**: every label $y_i = C(S_i, K_i, T_i, r_i, \sigma_i)$ is computed deterministically. This is in deliberate contrast to Monte Carlo estimates, which carry sampling noise proportional to $1/\sqrt{N_{\text{paths}}}$.

We draw the 500,000 parameter combinations using **uniform random sampling** across a realistic 5-dimensional financial parameter space.

### Parameter Ranges

| Parameter | Symbol | Range |
|---|---|---|
| Spot price | $S$ | [50, 200] |
| Strike price | $K$ | [50, 200] |
| Time to expiry | $T$ | [0.05, 3.0] (years) |
| Risk-free rate | $r$ | [0.0, 0.10] |
| Volatility | $\sigma$ | [0.05, 0.80] |


In [ ]:
N_SAMPLES = 500_000

# Parameter bounds: [S, K, T, r, sigma]
LOWER = np.array([50.0,  50.0,  0.05, 0.00, 0.05])
UPPER = np.array([200.0, 200.0, 3.00, 0.10, 0.80])

print(f'Generating {N_SAMPLES:,} samples via uniform random sampling ...')
t0 = time.time()

rng    = np.random.default_rng(seed=SEED)
params = rng.uniform(low=LOWER, high=UPPER, size=(N_SAMPLES, 5))

S     = params[:, 0]
K     = params[:, 1]
T     = params[:, 2]
r     = params[:, 3]
sigma = params[:, 4]

# Compute analytical BSM prices
prices = np.array([bsm_european(S[i], K[i], T[i], r[i], sigma[i]) for i in range(N_SAMPLES)])

print(f'  Done in {time.time()-t0:.1f}s')
print(f'  Price range: [{prices.min():.4f}, {prices.max():.4f}]')
print(f'  NaN count  : {np.isnan(prices).sum()}')


In [ ]:
df = pd.DataFrame({'S': S, 'K': K, 'T': T, 'r': r, 'sigma': sigma, 'price': prices})

# Drop degenerate near-zero prices from extreme OTM / low-T combinations
df = df[df['price'] > 1e-6].reset_index(drop=True)
print(f'Clean dataset size: {len(df):,} rows')
df.describe().round(4)

---
## 4. Preprocessing

Neural networks are sensitive to the scale of their inputs. The raw features span very different ranges: $S, K \in [50, 200]$ while $r \in [0, 0.10]$. Without normalisation, gradient descent update steps for large-scale features dominate, leading to slow and unstable convergence.

We apply **standardisation** (Z-score normalisation) via `StandardScaler`:

$$\hat{x}_j = \frac{x_j - \mu_j}{s_j}$$

where $\mu_j$ and $s_j$ are the mean and standard deviation of feature $j$ estimated on the **training set only** (to prevent data leakage).

In [ ]:
FEATURE_COLS = ['S', 'K', 'T', 'r', 'sigma']
TARGET_COL   = 'price'

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32).reshape(-1, 1)

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

# Fit scaler on training data ONLY
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Feature means (train): {scaler.mean_.round(4)}')
print(f'Feature stds  (train): {scaler.scale_.round(4)}')

---
## 5. DNN Architecture

Following the supervisor's guidance to keep the network **small and manageable**, we use:

- **Hidden layers**: 2
- **Neurons per layer**: 20
- **Activation**: ReLU, $\ell(z) = \max(z, 0)$. This non-polynomial, continuous function is the minimum requirement for the network to be a universal approximator (Leshno et al. 1993, cited in Davey & Zheng 2022).
- **Output**: 1 linear neuron — no activation, so the output is unbounded real-valued.
- **Initialisation**: Kaiming (He) normal, which sets weight variances to $2/n_{\text{in}}$ to compensate for ReLU halving the variance.

The forward pass for a network with $L = 2$ hidden layers is:
$$z_1 = A_1 x + b_1, \quad \tilde{z}_1 = \ell(z_1)$$
$$z_2 = A_2 \tilde{z}_1 + b_2, \quad \tilde{z}_2 = \ell(z_2)$$
$$\hat{C} = h(x) = A_3 \tilde{z}_2 + b_3$$

where $A_1 \in \mathbb{R}^{20 \times 5}$, $A_2 \in \mathbb{R}^{20 \times 20}$, $A_3 \in \mathbb{R}^{1 \times 20}$, and all $b_j$ are bias vectors of matching dimension.


In [ ]:
class OptionPricingDNN(nn.Module):
    """
    Small feedforward DNN for Black-Scholes call option price regression.
    Follows supervisor guidance: 1-2 hidden layers, 10-20 neurons each.

    Default architecture
    --------------------
    Input(5) -> FC(20)->ReLU -> FC(20)->ReLU -> FC(1)
    """

    def __init__(self, input_dim: int = 5, hidden_dims: list = None):
        super(OptionPricingDNN, self).__init__()

        if hidden_dims is None:
            hidden_dims = [20, 20]   # 2 hidden layers, 20 neurons each

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = OptionPricingDNN().to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')
print('\nNote: small network per supervisor guidance (2 hidden layers x 20 neurons).')


---
## 6. Training

### Loss Function — Mean Squared Error

$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{C}_i - C_i \right)^2$$

MSE heavily penalises large errors, encouraging the network to accurately price options across the full surface rather than sacrificing accuracy on tail cases.

### Optimiser — Adam

Adam (Kingma & Ba, 2015) combines momentum and adaptive per-parameter learning rates:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

where $\hat{m}_t$ and $\hat{v}_t$ are bias-corrected first and second moment estimates of the gradient. Adam converges faster than vanilla SGD on smooth financial surfaces.

A **ReduceLROnPlateau** scheduler halves the learning rate when validation loss plateaus for 5 consecutive epochs.

In [ ]:
BATCH_SIZE    = 4096
N_EPOCHS      = 60
LEARNING_RATE = 1e-3
VAL_SPLIT     = 0.10

X_tr = torch.tensor(X_train_sc, dtype=torch.float32)
y_tr = torch.tensor(y_train,    dtype=torch.float32)
X_te = torch.tensor(X_test_sc,  dtype=torch.float32)
y_te = torch.tensor(y_test,     dtype=torch.float32)

n_val   = int(VAL_SPLIT * len(X_tr))
n_train = len(X_tr) - n_val

train_ds = TensorDataset(X_tr[:n_train], y_tr[:n_train])
val_ds   = TensorDataset(X_tr[n_train:], y_tr[n_train:])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(f'Training samples  : {n_train:,}')
print(f'Validation samples: {n_val:,}')
print(f'Batch size        : {BATCH_SIZE}')
print(f'Epochs            : {N_EPOCHS}')
print(f'Batches / epoch   : {len(train_loader)}')

In [ ]:
def run_epoch(loader, model, criterion, optimizer, train: bool):
    """Run one full epoch. Returns mean MSE loss over batches."""
    model.train(train)
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)


train_losses, val_losses = [], []
best_val_loss    = float('inf')
best_model_state = None

print(f'{"Epoch":>6}  {"Train MSE":>12}  {"Val MSE":>12}  {"LR":>10}')
print('-' * 50)

t_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    tr_loss  = run_epoch(train_loader, model, criterion, optimizer, train=True)
    val_loss = run_epoch(val_loader,   model, criterion, optimizer, train=False)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'{epoch:>6}  {tr_loss:>12.6f}  {val_loss:>12.6f}  {lr_now:>10.2e}')

print(f'\nTraining complete in {time.time()-t_start:.1f}s  |  Best val MSE: {best_val_loss:.6f}')

model.load_state_dict(best_model_state)
model.to(DEVICE)
print('Best model weights restored.')

---
## 7. Evaluation

### 7.1 Loss Curves

A consistent, gap-free decrease in both training and validation loss confirms the network is learning the BSM surface without overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

epochs_range = range(1, N_EPOCHS + 1)
ax.semilogy(epochs_range, train_losses, label='Training MSE',   color='steelblue', linewidth=2)
ax.semilogy(epochs_range, val_losses,   label='Validation MSE', color='tomato',    linewidth=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss (log scale)', fontsize=12)
ax.set_title('Training and Validation Loss — DNN Black-Scholes Approximation', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: loss_curve.png')

### 7.2 Out-of-Sample Error Metrics

We evaluate on the held-out 20% test set using:

- **RMSE** $= \sqrt{\frac{1}{N}\sum_i(\hat{C}_i - C_i)^2}$ — in the same units as the option price; penalises large errors quadratically.
- **MAE** $= \frac{1}{N}\sum_i |\hat{C}_i - C_i|$ — robust to outliers; gives the typical absolute error.

In [ ]:
model.eval()

with torch.no_grad():
    y_pred_tensor = model(X_te.to(DEVICE)).cpu().numpy()

y_true = y_test.flatten()
y_pred = y_pred_tensor.flatten()

rmse    = np.sqrt(mean_squared_error(y_true, y_pred))
mae     = mean_absolute_error(y_true, y_pred)
rel_err = np.abs(y_pred - y_true) / (np.abs(y_true) + 1e-8)

print('=' * 45)
print('  Out-of-Sample Test Set Metrics')
print('=' * 45)
print(f'  RMSE                : {rmse:.6f}')
print(f'  MAE                 : {mae:.6f}')
print(f'  Mean Relative Error : {rel_err.mean()*100:.4f}%')
print(f'  Median Rel. Error   : {np.median(rel_err)*100:.4f}%')
print(f'  Max Absolute Error  : {np.abs(y_pred - y_true).max():.6f}')
print('=' * 45)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

idx = np.random.choice(len(y_true), size=5000, replace=False)

ax = axes[0]
ax.scatter(y_true[idx], y_pred[idx], alpha=0.15, s=6, color='steelblue')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel('BSM Analytical Price', fontsize=12)
ax.set_ylabel('DNN Predicted Price',  fontsize=12)
ax.set_title('Predicted vs Actual (5k sample)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

ax = axes[1]
residuals = y_pred - y_true
ax.hist(residuals, bins=100, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Residual (Predicted - Actual)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Residual Distribution', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation_plots.png')

---
## 8. Speed Benchmark — DNN vs Analytical BSM

One of the principal motivations for neural network surrogate models is inference speed. We benchmark pricing **100,000 options** using:

1. The analytical `bsm_european` function (sequential Python calls over a loop)
2. The trained DNN (single batched forward pass on CPU)

The DNN computes all prices simultaneously via a single matrix multiplication chain, whereas the Python loop incurs interpreter overhead per call.

In [ ]:
N_BENCH = 100_000

rng = np.random.default_rng(seed=99)
bench_params = rng.uniform(low=LOWER, high=UPPER, size=(N_BENCH, 5)).astype(np.float32)

# Benchmark 1: Analytical BSM (Python loop)
t0 = time.perf_counter()
bsm_prices = np.array([
    bsm_european(bench_params[i,0], bench_params[i,1],
                 bench_params[i,2], bench_params[i,3], bench_params[i,4])
    for i in range(N_BENCH)
])
t_bsm = time.perf_counter() - t0

# Benchmark 2: DNN (single batched forward pass, CPU)
bench_scaled = scaler.transform(bench_params)
bench_tensor = torch.tensor(bench_scaled, dtype=torch.float32)

model_cpu = model.to('cpu')
model_cpu.eval()

# Warm-up (avoids one-time JIT overhead in timing)
with torch.no_grad():
    _ = model_cpu(bench_tensor[:100])

t0 = time.perf_counter()
with torch.no_grad():
    dnn_prices = model_cpu(bench_tensor).numpy().flatten()
t_dnn = time.perf_counter() - t0

speedup = t_bsm / t_dnn

print('=' * 55)
print(f'  Benchmark: pricing {N_BENCH:,} options')
print('=' * 55)
print(f'  BSM analytical  : {t_bsm*1000:>9.2f} ms  ({t_bsm/N_BENCH*1e6:.2f} us/option)')
print(f'  DNN forward pass: {t_dnn*1000:>9.2f} ms  ({t_dnn/N_BENCH*1e6:.2f} us/option)')
print(f'  Speedup factor  : {speedup:>9.1f}x')
print('=' * 55)

corr = np.corrcoef(bsm_prices, dnn_prices)[0, 1]
print(f'  Pearson correlation (DNN vs BSM): {corr:.8f}')

In [ ]:
# Visualise benchmark results
fig, ax = plt.subplots(figsize=(7, 4))

methods = ['BSM Analytical\n(Python loop)', 'DNN Surrogate\n(batch forward)']
times   = [t_bsm * 1000, t_dnn * 1000]
colors  = ['tomato', 'steelblue']

bars = ax.bar(methods, times, color=colors, width=0.4, edgecolor='white')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{t:.1f} ms', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title(f'Pricing 100,000 Options — Speedup: {speedup:.1f}x', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: benchmark.png')

---
## 9. Pricing Surface Visualisation

To provide geometric intuition, we fix $T = 1.0$, $r = 0.05$, $\sigma = 0.20$ and plot the DNN-predicted call price surface over a $(S, K)$ grid alongside the exact BSM surface and the pointwise absolute error.

In [ ]:
T_fix, r_fix, sig_fix = 1.0, 0.05, 0.20
N_GRID = 80

S_grid = np.linspace(50, 200, N_GRID)
K_grid = np.linspace(50, 200, N_GRID)
SS, KK = np.meshgrid(S_grid, K_grid)

flat = np.column_stack([
    SS.ravel(), KK.ravel(),
    np.full(N_GRID**2, T_fix),
    np.full(N_GRID**2, r_fix),
    np.full(N_GRID**2, sig_fix)
]).astype(np.float32)

flat_sc = scaler.transform(flat)
flat_t  = torch.tensor(flat_sc, dtype=torch.float32)

with torch.no_grad():
    dnn_surface = model_cpu(flat_t).numpy().reshape(N_GRID, N_GRID)

bsm_surface = np.array([
    bsm_european(flat[i,0], flat[i,1], T_fix, r_fix, sig_fix)
    for i in range(N_GRID**2)
]).reshape(N_GRID, N_GRID)

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.1)

titles = ['BSM Analytical', 'DNN Approximation', 'Absolute Error |DNN - BSM|']
data   = [bsm_surface, dnn_surface, np.abs(bsm_surface - dnn_surface)]
cmaps  = ['viridis', 'viridis', 'hot_r']

for j, (surf, title, cmap) in enumerate(zip(data, titles, cmaps)):
    ax = fig.add_subplot(gs[j], projection='3d')
    ax.plot_surface(SS, KK, surf, cmap=cmap, alpha=0.9, linewidth=0)
    ax.set_xlabel('S', fontsize=9)
    ax.set_ylabel('K', fontsize=9)
    ax.set_zlabel('C', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.view_init(elev=28, azim=-60)

fig.suptitle(f'Call Price Surface  (T={T_fix}, r={r_fix}, sigma={sig_fix})',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('pricing_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pricing_surface.png')

---
## 10. Save the Trained Model

We persist both the network weights and the `StandardScaler` so the surrogate can be loaded and used without retraining.

In [ ]:
import joblib

torch.save(model.state_dict(), 'dnn_bs_model.pt')
joblib.dump(scaler, 'feature_scaler.pkl')

print('Saved model  : dnn_bs_model.pt')
print('Saved scaler : feature_scaler.pkl')

# Load and sanity-check
loaded_model = OptionPricingDNN()
loaded_model.load_state_dict(torch.load('dnn_bs_model.pt', map_location='cpu'))
loaded_model.eval()

test_params = [[100.0, 100.0, 1.0, 0.05, 0.20]]
with torch.no_grad():
    inp     = torch.tensor(scaler.transform(test_params), dtype=torch.float32)
    dnn_val = loaded_model(inp).item()

bsm_val = bsm_european(100.0, 100.0, 1.0, 0.05, 0.20)

print(f'\nSanity check  S=100, K=100, T=1, r=5%, sigma=20%')
print(f'  BSM analytical : {bsm_val:.6f}')
print(f'  DNN loaded     : {dnn_val:.6f}')
print(f'  Absolute error : {abs(dnn_val - bsm_val):.6f}')

---
## 3. Data Generation — Analytical Ground Truth

Because the BSM formula is exact, we generate training data with **zero variance**: every label $y_i = C(S_i, K_i, T_i, r_i, \sigma_i)$ is computed deterministically. This is in deliberate contrast to Monte Carlo estimates, which carry sampling noise proportional to $1/\sqrt{N_{\text{paths}}}$.

We draw the 500,000 parameter combinations using **uniform random sampling** across a realistic 5-dimensional financial parameter space.

### Parameter Ranges

| Parameter | Symbol | Range |
|---|---|---|
| Spot price | $S$ | [50, 200] |
| Strike price | $K$ | [50, 200] |
| Time to expiry | $T$ | [0.05, 3.0] (years) |
| Risk-free rate | $r$ | [0.0, 0.10] |
| Volatility | $\sigma$ | [0.05, 0.80] |
